# 01 · Carga de datos y variable objetivo

**Objetivo:** cargar los microdatos de la Encuesta de Estructura Salarial (INE) de 2018 y 2022, construir la variable objetivo (salario bruto anual) y comprobar que coincide con las cifras oficiales publicadas por el INE.

Fuente: INE, www.ine.es.

In [1]:
import sys
from pathlib import Path

# Localiza la raíz del proyecto (la carpeta que contiene src/) desde donde se abra el notebook
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(RAIZ))

import pandas as pd
from src.etiquetas import aplicar_etiquetas, NOMBRES
from src.datos import (
    ANIOS, MEDIA_PUBLICADA_INE, DATA_PROCESSED,
    leer_ees, calcular_salario_anual, dias_anio_computables,
    media_ponderada, cargar_ees_unificada,
)

pd.set_option("display.max_columns", 60)

## 1. Lectura de los dos años

In [2]:
ees18 = leer_ees(2018)
ees22 = leer_ees(2022)

print(f"2018: {ees18.shape[0]:,} trabajadores x {ees18.shape[1]} columnas")
print(f"2022: {ees22.shape[0]:,} trabajadores x {ees22.shape[1]} columnas")

2018: 216,726 trabajadores x 57 columnas
2022: 240,490 trabajadores x 57 columnas


## 2. Comprobación de que ambos años son compatibles

In [3]:
print("Columnas solo en 2018:", sorted(set(ees18.columns) - set(ees22.columns)))
print("Columnas solo en 2022:", sorted(set(ees22.columns) - set(ees18.columns)))

tipos_distintos = [c for c in ees18.columns if c in ees22.columns and ees18[c].dtype != ees22[c].dtype]
print("Columnas con tipo distinto:", tipos_distintos)
print("Valores nulos 2018:", int(ees18.isna().sum().sum()), "| 2022:", int(ees22.isna().sum().sum()))

Columnas solo en 2018: []
Columnas solo en 2022: []
Columnas con tipo distinto: []
Valores nulos 2018: 0 | 2022: 0


In [4]:
# Las categorías de las variables clave son las mismas en los dos años
clave = ["NUTS1", "CNACE", "ESTRATO2", "SEXO", "TIPOPAIS", "CNO1", "ESTU", "ANOS2"]
pd.DataFrame({
    "categorias_2018": [ees18[c].nunique() for c in clave],
    "categorias_2022": [ees22[c].nunique() for c in clave],
    "mismos_codigos": [set(ees18[c]) == set(ees22[c]) for c in clave],
}, index=clave)

,categorias_2018,categorias_2022,mismos_codigos
NUTS1,7,7,True
CNACE,27,27,True
ESTRATO2,5,5,True
SEXO,2,2,True
TIPOPAIS,2,2,True
CNO1,17,17,True
ESTU,7,7,True
ANOS2,6,6,True


## 3. Variable objetivo: salario bruto anual

Siguiendo el documento del INE *"Obtención de los resultados publicados"*:

- **Salario bruto** = `RETRINOIN + RETRIIN + VESPNOIN + VESPIN` (retribuciones ordinarias y en especie, derivadas o no de incapacidad temporal).
- **Días computables** = `DRELABAM × 30,42 + DRELABAD − DSIESPA2 − DSIESPA4`.
- **Salario anual** = salario bruto × 365 / días computables (se eleva a año completo para quien no trabajó todo el año).
- Las medias se calculan **ponderando con `FACTOTAL`** (factor de elevación), para que representen al total de asalariados.

In [5]:
for df in (ees18, ees22):
    df["dias_computables"] = dias_anio_computables(df)
    df["salario_anual"] = calcular_salario_anual(df)

ees22[["DRELABAM", "DRELABAD", "dias_computables", "RETRINOIN", "VESPNOIN", "salario_anual", "FACTOTAL"]].head()

,DRELABAM,DRELABAD,dias_computables,RETRINOIN,VESPNOIN,salario_anual,FACTOTAL
0,12,0,253.04,6615.01,0.0,11177.374921,17.55
1,12,0,365.04,30757.32,0.0,30753.949704,17.55
2,12,0,365.04,7739.92,0.0,7927.521231,17.55
3,12,0,365.04,12407.11,0.0,12597.439459,17.55
4,12,0,365.04,19717.33,0.0,19839.965757,17.55


## 4. Validación contra las cifras oficiales del INE

In [6]:
validacion = pd.DataFrame({
    "media_calculada": [media_ponderada(df["salario_anual"], df["FACTOTAL"]) for df in (ees18, ees22)],
    "media_publicada_INE": [MEDIA_PUBLICADA_INE[a] for a in ANIOS],
}, index=ANIOS)
validacion["diferencia_%"] = 100 * (validacion["media_calculada"] / validacion["media_publicada_INE"] - 1)
validacion.round(2)

,media_calculada,media_publicada_INE,diferencia_%
2018,24006.89,24009.12,-0.01
2022,26946.31,26948.87,-0.01


La media calculada coincide con la publicada por el INE con una diferencia inferior al 0,02 %, debida al redondeo. **La variable objetivo está bien construida.**

## 5. Unión de los dos años

In [7]:
ees = cargar_ees_unificada()
print(f"Dataset unificado: {ees.shape[0]:,} filas x {ees.shape[1]} columnas")
ees.groupby("anio")["salario_anual"].describe().round(0)

Dataset unificado: 457,216 filas x 58 columnas


,count,mean,std,min,25%,50%,75%,max
anio,,,,,,,,
2018,216726.0,28482.0,34999.0,135.0,16162.0,22815.0,33930.0,4225998.0
2022,240490.0,31025.0,32138.0,187.0,17599.0,24769.0,36921.0,2794902.0


## 6. Etiquetas legibles

Los códigos del INE (`CNO1 = "C0"`, `ESTU = "7"`...) se traducen a texto con el diseño de registro `dr_EES_2022.xlsx`. Se conservan las columnas originales y se añade una columna legible por variable (`ocupacion`, `estudios`, `region`...).

In [8]:
ees = aplicar_etiquetas(ees)
ees[list(NOMBRES.values())].head()

,region,sector,tamano_empresa,control,mercado,convenio,sexo,nacionalidad,ocupacion,responsabilidad,estudios,jornada,contrato,edad
0,Noroeste,Transporte de viajeros y mercancías,De 1 a 49 trabajadores,Privado,Unión Europea,Convenio sectorial de ámbito inferior,Hombre,España,Conductores y operadores de maquinaria móvil,Sin responsabilidad,Secundaria 1ª etapa (ESO),Tiempo parcial,Temporal,60 o más
1,Noroeste,Transporte de viajeros y mercancías,De 1 a 49 trabajadores,Privado,Unión Europea,Convenio sectorial de ámbito inferior,Hombre,España,Conductores y operadores de maquinaria móvil,Sin responsabilidad,Secundaria 1ª etapa (ESO),Tiempo completo,Indefinido,60 o más
2,Noroeste,Transporte de viajeros y mercancías,De 1 a 49 trabajadores,Privado,Unión Europea,Convenio sectorial de ámbito inferior,Mujer,España,Otros técnicos y profesionales científicos,Sin responsabilidad,"Licenciatura, grado, máster o doctorado",Tiempo completo,Indefinido,40 a 49
3,Noroeste,Transporte de viajeros y mercancías,De 1 a 49 trabajadores,Privado,Unión Europea,Convenio sectorial de ámbito inferior,Hombre,España,Conductores y operadores de maquinaria móvil,Sin responsabilidad,Secundaria 1ª etapa (ESO),Tiempo completo,Indefinido,60 o más
4,Noroeste,Transporte de viajeros y mercancías,De 1 a 49 trabajadores,Privado,Unión Europea,Convenio sectorial de ámbito inferior,Hombre,España,Conductores y operadores de maquinaria móvil,Sin responsabilidad,Secundaria 1ª etapa (ESO),Tiempo completo,Indefinido,40 a 49


In [9]:
# Ejemplo: distribución de trabajadores por ocupación en 2022 (sin ponderar)
ees.loc[ees.anio == 2022, "ocupacion"].value_counts()

ocupacion
Técnicos y profesionales de apoyo                              39199
Otros técnicos y profesionales científicos                     28666
Trabajadores cualificados de la industria                      22405
Empleados de oficina con atención al público                   17240
Profesionales de la salud y la enseñanza                       16793
Peones de agricultura, industria, construcción y transporte    15346
Trabajadores no cualificados en servicios                      15339
Empleados de oficina sin atención al público                   15130
Trabajadores de restauración y comercio                        14917
Trabajadores de salud y cuidado de personas                    14597
Operadores de instalaciones y montadores                       10544
Conductores y operadores de maquinaria móvil                    9963
Directores y gerentes                                           7905
Trabajadores cualificados de la construcción                    5936
Trabajadores de protecci

**Nota sobre el tamaño de empresa:** los códigos `0` y `4` de `ESTRATO2` no son un tamaño concreto, sino agrupaciones que el INE aplica para proteger el anonimato (≈2,4 % de los registros). Se tratarán en la fase de preparación.

In [10]:
ees["tamano_empresa"].value_counts()

tamano_empresa
200 o más trabajadores                206685
De 1 a 49 trabajadores                128296
De 50 a 199 trabajadores              111062
50 o más (sin desagregar)               8535
Sin desagregar (todos los tamaños)      2638
Name: count, dtype: int64

## 7. Guardado

In [11]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
ees.to_parquet(DATA_PROCESSED / "ees_2018_2022.parquet", index=False)
print("Guardado en", DATA_PROCESSED / "ees_2018_2022.parquet")

Guardado en /home/claude/tfm-salarios-ees/data/processed/ees_2018_2022.parquet


## Pendiente

- Deflactar el salario de 2018 a euros de 2022 con el IPC.